In [3]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel

def remap_severity(severity_series):
    """
    Remap collision severity values to a reversed scale and convert to an ordered categorical variable.
    
    The function converts severity scores from the original scale to a reversed scale as follows:
        1 → 4
        2 → 3
        3 → 2
        4 → 1
        
    Parameters:
    severity_series (pd.Series): Series containing collision severity values as strings or numbers.
    
    Returns:
    pd.Categorical: An ordered categorical series with severity levels [1, 2, 3, 4],
                    where 1 represents the least severe and 4 the most severe.
    """
    # Convert severity values to numeric, coercing errors to NaN
    severity_numeric = pd.to_numeric(severity_series, errors="coerce")
    # Define the mapping from original to reversed scale
    severity_mapping = {1: 4, 2: 3, 3: 2, 4: 1}
    remapped = severity_numeric.map(severity_mapping)
    # Define the order for the categorical variable
    severity_order = [1, 2, 3, 4]
    return pd.Categorical(remapped, categories=severity_order, ordered=True)

def categorize_time_of_day(time_str):
    """
    Categorize a time (given as a string) into a time-of-day group.
    
    The function interprets the input as a time in HHMM format and assigns:
        "M" for Morning (06:00 - 11:59),
        "A" for Afternoon (12:00 - 17:59),
        "E" for Evening (18:00 - 23:59),
        "N" for Night (00:00 - 05:59).
    
    Parameters:
    time_str (str): A string representing the time in HHMM format (e.g., "0600" for 6:00 AM).
    
    Returns:
    str: A single character representing the time of day ("M", "A", "E", or "N").
         If the input is invalid, returns np.nan.
    """
    try:
        time_int = int(time_str)
        if 600 <= time_int < 1200:
            return "M"  # Morning
        elif 1200 <= time_int < 1800:
            return "A"  # Afternoon
        elif 1800 <= time_int < 2400:
            return "E"  # Evening
        elif 0 <= time_int < 600:
            return "N"  # Night
    except (ValueError, TypeError):
        return np.nan

# Load dataset
file_path = "Crashes.csv"  # Update with the actual file path
df = pd.read_csv(file_path, dtype=str)

# Keep only relevant columns
columns_to_keep = ["COLLISION_SEVERITY", "DAY_OF_WEEK", "COLLISION_TIME"]
df = df[columns_to_keep]

# Apply severity remapping using the dedicated function
df["COLLISION_SEVERITY"] = remap_severity(df["COLLISION_SEVERITY"])

# Create TIME_OF_DAY by categorizing COLLISION_TIME and convert to a categorical variable
df["TIME_OF_DAY"] = df["COLLISION_TIME"].apply(categorize_time_of_day).astype("category")
df.drop(columns=["COLLISION_TIME"], inplace=True)

# Collapse DAY_OF_WEEK into "Weekday" (Mon-Fri) vs. "Weekend" (Sat-Sun)
df["DAY_OF_WEEK"] = df["DAY_OF_WEEK"].apply(lambda x: "Weekday" if x in ["1", "2", "3", "4", "5"] else "Weekend")




FileNotFoundError: [Errno 2] No such file or directory: 'Crashes.csv'

In [2]:
# Apply dummy encoding, keeping "Weekday" as the reference category
df = pd.get_dummies(df, columns=["DAY_OF_WEEK", "TIME_OF_DAY"], drop_first=True)

# Define independent variables (all columns except the target)
independent_vars = df.columns.difference(["COLLISION_SEVERITY"])

# Specify and calibrate the Ordered Logit Model
model = OrderedModel(
    df["COLLISION_SEVERITY"],
    df[independent_vars],
    distr="logit"
)

result = model.fit(method="bfgs")
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.933109
         Iterations: 25
         Function evaluations: 27
         Gradient evaluations: 27
                             OrderedModel Results                             
Dep. Variable:     COLLISION_SEVERITY   Log-Likelihood:                -12531.
Model:                   OrderedModel   AIC:                         2.508e+04
Method:            Maximum Likelihood   BIC:                         2.513e+04
Date:                Thu, 27 Feb 2025                                         
Time:                        12:39:13                                         
No. Observations:               13429                                         
Df Residuals:                   13422                                         
Df Model:                           4                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

In [3]:
# Compute and display Odds Ratios
odds_ratios = np.exp(result.params)
print("\nOdds Ratios:\n", odds_ratios)

# Predict probabilities for each severity level
predicted_probs = result.predict()

# Convert predictions to a DataFrame
predicted_probs_df = pd.DataFrame(predicted_probs)

# Determine the most likely severity level for each observation
df["Predicted_Severity"] = predicted_probs_df.idxmax(axis=1)

# Display performance metrics with a confusion matrix (objective results only)
confusion_matrix = pd.crosstab(df["COLLISION_SEVERITY"], df["Predicted_Severity"],
                               rownames=['Actual'], colnames=['Predicted'])
print("\nConfusion Matrix:\n", confusion_matrix)



Odds Ratios:
 DAY_OF_WEEK_Weekend    1.179909
TIME_OF_DAY_E          1.360871
TIME_OF_DAY_M          0.964779
TIME_OF_DAY_N          2.066064
1/2                    1.650865
2/3                    2.014591
3/4                    1.871090
dtype: float64

Confusion Matrix:
 Predicted     0    1
Actual              
1          7496  264
2          4237  212
3           905  109
4           175   31
